In [6]:
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from Lib_classes.DataFrameFilesBuilderStack import DataFrameStackBuilder
from Lib_classes.FileArchitecture import PdPickleFileArchitecture
from Lib_classes.funcs import raise_none
from Lib_classes.Paths import Paths
from Source.Source_aggregate.SourceAggregator import SourceAggregator

In [2]:
path = os.path.join(Paths.dir_dataframe_pickles, 'Bearing', 'train')  # noqa: F821
def split_column(x):
    x = str(Path(x).parts[len(Path(Paths.dir_source).absolute().parts)])
    match = re.search(r'^([\D\d]+\D)\d+$', x) 
    if match is None:
        return x
    return match.group(1)

get_split_column = lambda df:df['original_path'].map(split_column)
folder = os.path.join(path,'splits')
chunk_size = 1000

In [3]:

pfa = PdPickleFileArchitecture.open(path)

In [7]:
inx = defaultdict(list)

for i in  pfa.iter_stack_index():
    df = pfa.get_file(i)
    df['split'] = get_split_column(df)
    for k, vs in df.groupby('split').groups.items():
        for v in vs:
            inx[k].append((i, v))
        

In [8]:
inx

defaultdict(list,
            {'CWRU': [(0, 23),
              (0, 31),
              (0, 35),
              (0, 36),
              (0, 60),
              (0, 71),
              (0, 75),
              (0, 76),
              (0, 81),
              (0, 164),
              (0, 181),
              (0, 196),
              (0, 207),
              (0, 216),
              (0, 219),
              (0, 365),
              (0, 380),
              (0, 451),
              (0, 462),
              (0, 464),
              (0, 484),
              (0, 507),
              (0, 517),
              (0, 576),
              (0, 584),
              (0, 585),
              (0, 619),
              (0, 631),
              (0, 656),
              (0, 660),
              (0, 670),
              (0, 722),
              (0, 745),
              (0, 751),
              (0, 764),
              (0, 766),
              (0, 805),
              (0, 841),
              (0, 850),
              (0, 856),
              (0, 925),

In [9]:
class ProviderChunksMixin:
    def __init__(self, 
                 pfa: PdPickleFileArchitecture,
                 rng: np.random.Generator
        ):
        self.pfa = pfa
        self._rng = rng
        
    def provide_chunks(self, chunks:dict):
        for key, chunk in chunks.items():
            chunk_df = None
            for i in self.pfa.iter_stack_index():
                cur_chunks = list(map(lambda x: x[1], filter(lambda x: x[0] == i, chunk)))
                df = self.pfa.get_file(i).loc[cur_chunks]
                chunk_df = pd.concat([chunk_df, df], ignore_index=True) if chunk_df is not None else df
            chunk_df = raise_none(chunk_df).sample(frac=1, random_state=self._rng.integers(0, np.iinfo(np.int32).max)).reset_index(drop=True)
            yield key, chunk_df

In [10]:
pcm = ProviderChunksMixin(pfa, np.random.default_rng(0))

In [11]:
for k, df in tqdm(pcm.provide_chunks(inx), total=len(inx.items())):
    path_ = os.path.join(folder, k)
    Paths.make_path(path_)
    with DataFrameStackBuilder(
        list(pfa.get_columns()), 
        PdPickleFileArchitecture(
            path_,
            max_stack_size=chunk_size
        )
    ) as pmf:
        pmf.add_df(df)

100%|██████████| 9/9 [01:36<00:00, 10.69s/it]


In [12]:
pfa = PdPickleFileArchitecture.open(path)

In [13]:
builders = {}
output_base = Path(folder)
columns = list(pfa.get_columns())
rng = np.random.default_rng(0)

try:
    for i in tqdm(pfa.iter_stack_index(), desc="Processing Source Files", total=len(list(pfa.iter_stack_index()))):
        full_df = pfa.get_file(i)
        full_df['split'] = get_split_column(full_df)
        grouped = full_df.groupby('split')
        
        for split_name, group_df in grouped:
            
            if split_name not in builders:
                split_dir = output_base / str(split_name)
                Paths.make_path(str(split_dir))
                
                builders[split_name] = DataFrameStackBuilder(
                    columns,
                    PdPickleFileArchitecture(str(split_dir), max_stack_size=chunk_size)
                )
                builders[split_name].__enter__()
            group_df = group_df.sample(frac=1, random_state=rng.integers(0, np.iinfo(np.int32).max)).reset_index(drop=True)
            builders[split_name].add_df(group_df)

finally:
    for builder in builders.values():
        builder.__exit__(None, None, None)

Processing Source Files: 100%|██████████| 6/6 [00:22<00:00,  3.83s/it]


In [16]:
path =  os.path.join(Paths.dir_dataframe_pickles, 'Bearing', 'validate')
pfa = PdPickleFileArchitecture.open(path)
SourceAggregator.split_groups(pfa)

Processing Source Files: 100%|██████████| 1/1 [00:02<00:00,  2.02s/it]
